## Tabular Data Observations

### General

- Dataset contains 3693 listings and 79 columns
- There's a mixture of numerical, categorical, textual, and spatial features
- Latitude and longitude are available for all listings and look useable for spatial analysis, but will need to be compared against the .geojson file provided

### Price

- `price` is the target variable
- Stored as strings (e.g. '$104.00'), not numerical values
- Some prices contain comma seperators (e.g. '$1,250.00')
- Approx. 28% of rows (1050 listings) have missing prices
- Remaining prices appear cleanable through string preprocessing
- Price distribution appears heavily right-skewed
- Maximum observed price is 13,000, suggesting the presence of luxury-property outliers
- Minimum price observed is 1, should be investigated further

### Missing Data

- Several review-related features contain significant amounts of missing data
- Neighborhood-related text fields text fieds contain significant amounts of missing data
- Missing prices seemingly don't correspond exclusively to unavailable listings

### Availability

- `has_availability` is encoded as:
  - `t` = available
  - `NaN` = unavailable
- Listings with missing `has_availability` have zero availability across all observed booking windows (`availability_30`, `availability_60`, `availability_90`, `availability_365`).
- The column can therefore be treated as a boolean feature during preprocessing

### Feature Selection

Likely drop:
- IDs
- URLs
- Scraping metadata
- Image URLs

Requires discussion:
- Text fields (`description`, `host_about`, `amenities`)
- Review score features
- Host-related metadata
- Revenue and occupancy estimates (possible leakage)

Likely keep:
- Spatial features (`latitude`, `longitude`)
- Property characteristics
- Room type
- Availability metrics
- Review counts
- Host verification/superhost information


In [2]:
import pandas as pd
import geopandas as gpd
from src.utils.paths import RAW_DATA_DIR

In [3]:
df = pd.read_csv(RAW_DATA_DIR / "listings.csv")
gdf = gpd.read_file(RAW_DATA_DIR / "neighbourhoods.geojson")

In [4]:
df.head()


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,71609,https://www.airbnb.com/rooms/71609,20250928035118,2025-09-28,previous scrape,Ensuite Room (Room 1 & 2) near EXPO,For 3 rooms.Book room 1&2 and room 4,NaN,https://a0.muscache.com/pictures/24453191/3580...,367042,...,4.78,4.26,4.32,NaN,f,5,0,5,0,0.11
1,71896,https://www.airbnb.com/rooms/71896,20250928035118,2025-09-28,city scrape,B&B Room 1 near Airport & EXPO,NaN,NaN,https://a0.muscache.com/pictures/2440674/ac4f4...,367042,...,4.43,4.17,4.04,NaN,f,5,0,5,0,0.14
2,71903,https://www.airbnb.com/rooms/71903,20250928035118,2025-09-28,city scrape,Room 2-near Airport & EXPO,"Like your own home, 24hrs access.",Quiet and view of the playground with exercise...,https://a0.muscache.com/pictures/568743/7bc623...,367042,...,4.64,4.50,4.36,NaN,f,5,0,5,0,0.26
3,275343,https://www.airbnb.com/rooms/275343,20250928035118,2025-09-28,previous scrape,10min walk to MRT & a Cozy Room with window! (1),**IMPORTANT NOTES: READ BEFORE YOU BOOK! <br ...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,1439258,...,4.42,4.53,4.63,S0399,t,49,0,49,0,0.13
4,275344,https://www.airbnb.com/rooms/275344,20250928035118,2025-09-28,previous scrape,15 mins to Outram MRT Single Room (2),Lovely home for the special guest !,Bus stop <br />Food center <br />Supermarket,https://a0.muscache.com/pictures/miso/Hosting-...,1439258,...,4.50,4.63,4.13,S0399,t,49,0,49,0,0.10


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3693 entries, 0 to 3692
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            3693 non-null   int64  
 1   listing_url                                   3693 non-null   str    
 2   scrape_id                                     3693 non-null   int64  
 3   last_scraped                                  3693 non-null   str    
 4   source                                        3693 non-null   str    
 5   name                                          3693 non-null   str    
 6   description                                   3622 non-null   str    
 7   neighborhood_overview                         1720 non-null   str    
 8   picture_url                                   3693 non-null   str    
 9   host_id                                       3693 non-null   int64  
 10 

In [6]:
df.isna().sum().sort_values(ascending=False)

calendar_updated                                3693
neighbourhood                                   1973
neighborhood_overview                           1973
review_scores_value                             1847
review_scores_location                          1847
                                                ... 
instant_bookable                                   0
calculated_host_listings_count                     0
calculated_host_listings_count_entire_homes        0
calculated_host_listings_count_private_rooms       0
calculated_host_listings_count_shared_rooms        0
Length: 79, dtype: int64

In [7]:
df.describe()

,id,scrape_id,host_id,host_listings_count,host_total_listings_count,latitude,longitude,accommodates,bathrooms,bedrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,3.693000e+03,3.693000e+03,3.693000e+03,3693.000000,3693.000000,3693.000000,3693.000000,3693.000000,2841.000000,3381.000000,...,1847.000000,1847.000000,1847.000000,1846.000000,1846.000000,3693.000000,3693.000000,3693.000000,3693.000000,1847.000000
mean,6.036958e+17,2.025093e+13,2.076744e+08,68.451665,117.102627,1.310000,103.844362,2.668291,1.408307,1.248151,...,4.511316,4.739626,4.701473,4.687741,4.428196,41.066071,29.327376,11.385594,0.141348,0.585246
std,6.053931e+17,3.906779e-03,2.179386e+08,128.656518,228.090618,0.030690,0.043068,1.752777,0.911075,0.778559,...,0.642694,0.474100,0.556684,0.463618,0.648710,57.836939,58.372559,19.649205,1.109671,1.609855
min,7.160900e+04,2.025093e+13,2.182320e+05,1.000000,1.000000,1.222720,103.634570,1.000000,0.000000,0.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.010000
25%,3.349064e+07,2.025093e+13,3.223219e+07,3.000000,4.000000,1.291890,103.834080,2.000000,1.000000,1.000000,...,4.285000,4.670000,4.670000,4.550000,4.140000,3.000000,0.000000,1.000000,0.000000,0.040000
50%,6.388900e+17,2.025093e+13,1.386492e+08,11.000000,16.000000,1.304490,103.844100,2.000000,1.000000,1.000000,...,4.730000,4.930000,4.930000,4.860000,4.600000,11.000000,1.000000,3.000000,0.000000,0.170000
75%,1.245009e+18,2.025093e+13,3.007659e+08,69.000000,99.000000,1.317356,103.860173,4.000000,2.000000,1.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,59.000000,26.000000,12.000000,0.000000,0.520000
max,1.518509e+18,2.025093e+13,7.196559e+08,517.000000,885.000000,1.457802,103.989570,16.000000,13.000000,8.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,219.000000,219.000000,89.000000,13.000000,26.070000


In [8]:
df.columns.tolist()

['id',
 'listing_url',
 'scrape_id',
 'last_scraped',
 'source',
 'name',
 'description',
 'neighborhood_overview',
 'picture_url',
 'host_id',
 'host_url',
 'host_name',
 'host_since',
 'host_location',
 'host_about',
 'host_response_time',
 'host_response_rate',
 'host_acceptance_rate',
 'host_is_superhost',
 'host_thumbnail_url',
 'host_picture_url',
 'host_neighbourhood',
 'host_listings_count',
 'host_total_listings_count',
 'host_verifications',
 'host_has_profile_pic',
 'host_identity_verified',
 'neighbourhood',
 'neighbourhood_cleansed',
 'neighbourhood_group_cleansed',
 'latitude',
 'longitude',
 'property_type',
 'room_type',
 'accommodates',
 'bathrooms',
 'bathrooms_text',
 'bedrooms',
 'beds',
 'amenities',
 'price',
 'minimum_nights',
 'maximum_nights',
 'minimum_minimum_nights',
 'maximum_minimum_nights',
 'minimum_maximum_nights',
 'maximum_maximum_nights',
 'minimum_nights_avg_ntm',
 'maximum_nights_avg_ntm',
 'calendar_updated',
 'has_availability',
 'availability_30

In [9]:
df["price"].head(10)

0        NaN
1    $104.00
2     $76.00
3        NaN
4        NaN
5     $85.00
6        NaN
7        NaN
8     $41.00
9    $101.00
Name: price, dtype: str

In [10]:
df["price"].isna().sum()

np.int64(1050)

In [11]:
df[df["price"].isna()][
    [
        "room_type",
        "availability_365",
        "number_of_reviews",
        "has_availability"
    ]
].head(20)

,room_type,availability_365,number_of_reviews,has_availability
0,Private room,90,19,t
3,Private room,0,20,t
4,Private room,0,16,t
6,Private room,0,17,t
7,Private room,0,5,t
11,Private room,83,27,t
13,Private room,0,0,NaN
14,Private room,0,75,NaN
15,Private room,0,0,NaN
16,Private room,89,26,t


In [12]:
df["has_availability"].value_counts()

has_availability
t    3415
Name: count, dtype: int64

In [13]:
df[df["price"].isna()]["has_availability"].value_counts()

has_availability
t    786
Name: count, dtype: int64

In [14]:
df["price"].dropna().head(20)

1     $104.00
2      $76.00
5      $85.00
8      $41.00
9     $101.00
10     $91.00
12     $53.00
18    $127.00
19    $127.00
21     $47.00
22     $67.00
26    $172.00
28     $76.00
29    $222.00
30     $82.00
31     $85.00
33     $77.00
34    $549.00
35     $76.00
36     $66.00
Name: price, dtype: str

In [15]:
df["price"].dropna().str.contains(",", regex=False).sum()

np.int64(85)

In [16]:
price_clean = (df["price"]
               .str.replace("$", "", regex=False)).str.replace(",", "", regex=False).astype(float)

In [17]:
price_clean.describe()

count     2643.000000
mean       345.269769
std        842.116630
min          1.000000
25%         94.000000
50%        221.000000
75%        359.000000
max      13000.000000
Name: price, dtype: float64

In [18]:
df["has_availability"].unique()

<StringArray>
['t', nan]
Length: 2, dtype: str

In [19]:
df["has_availability"].isna().sum()

np.int64(278)

In [20]:
df[df["has_availability"].isna()][
    [
        "availability_30",
        "availability_60",
        "availability_90",
        "availability_365"
    ]
].head()

,availability_30,availability_60,availability_90,availability_365
13,0,0,0,0
14,0,0,0,0
15,0,0,0,0
17,0,0,0,0
25,0,0,0,0


In [21]:
sorted(df["neighbourhood_cleansed"].dropna().unique())

['Ang Mo Kio',
 'Bedok',
 'Bishan',
 'Bukit Batok',
 'Bukit Merah',
 'Bukit Panjang',
 'Bukit Timah',
 'Central Water Catchment',
 'Changi',
 'Choa Chu Kang',
 'Clementi',
 'Downtown Core',
 'Geylang',
 'Hougang',
 'Jurong East',
 'Jurong West',
 'Kallang',
 'Mandai',
 'Marina South',
 'Marine Parade',
 'Museum',
 'Newton',
 'Novena',
 'Orchard',
 'Outram',
 'Pasir Ris',
 'Pioneer',
 'Punggol',
 'Queenstown',
 'River Valley',
 'Rochor',
 'Sembawang',
 'Sengkang',
 'Serangoon',
 'Singapore River',
 'Southern Islands',
 'Sungei Kadut',
 'Tampines',
 'Tanglin',
 'Toa Payoh',
 'Tuas',
 'Western Water Catchment',
 'Woodlands',
 'Yishun']

In [22]:
set(df["neighbourhood_cleansed"].dropna()) == set(gdf["neighbourhood"])

False

In [23]:
set(df["neighbourhood_cleansed"].dropna()) - set(gdf["neighbourhood"])

set()

In [24]:
set(gdf["neighbourhood"]) - set(df["neighbourhood_cleansed"].dropna())

{'Boon Lay',
 'Changi Bay',
 'Lim Chu Kang',
 'Marina East',
 'North-Eastern Islands',
 'Paya Lebar',
 'Seletar',
 'Simpang',
 'Straits View',
 'Tengah',
 'Western Islands'}